In [ ]:
pip install kagglehub lime scikit-learn pandas numpy matplotlib

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np

In [ ]:
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "uciml/german-credit",
    "german_credit_data.csv"
)
df = df.drop(columns=["Unnamed: 0"])

In [ ]:
target = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data",
    sep=" ",
    header=None
)
y = target.iloc[:, -1].map({1: 1, 2: 0})
df = df.iloc[:len(y)]

In [ ]:
X = df
categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=200, random_state=42)

In [ ]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model)
])
pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
X_train_transformed = pipeline.named_steps["preprocess"].transform(X_train)
feature_names = (
    numerical_cols.tolist() +
    pipeline.named_steps["preprocess"]
    .named_transformers_["cat"]
    .get_feature_names_out(categorical_cols)
    .tolist()
)
explainer = LimeTabularExplainer(
    training_data=X_train_transformed if isinstance(X_train_transformed, np.ndarray) else X_train_transformed.toarray(),
    feature_names=feature_names,
    class_names=["Bad Credit", "Good Credit"],
    mode="classification"
)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

test_instances = X_test.iloc[:5]

for i in range(5):
    instance = pipeline.named_steps["preprocess"].transform(test_instances.iloc[[i]])
    if hasattr(instance, "toarray"):
        instance = instance.toarray()[0]
    else:
        instance = instance[0]

    exp = explainer.explain_instance(
        instance,
        pipeline.named_steps["model"].predict_proba,
        num_features=10
    )

    fig = exp.as_pyplot_figure(label=1)
    fig.suptitle(f"LIME Explanation – Test Instance {i+1}", fontweight="bold")
    plt.tight_layout()
    plt.show()
